# GOM NHÓM DỮ LIỆU VN30 THEO SIÊU TÀI SẢN NGÀNH (CÁCH 1 - EQUAL-WEIGHTED SECTOR INDEX)

### Mục tiêu nghiên cứu (Đồ án UIT - Tối ưu hóa & Ứng dụng):
1. **Giải quyết vấn đề chiều quyết định (Curse of Dimensionality)**:
   - Giảm từ **30 mã riêng lẻ** xuống **7 nhóm ngành đại diện** ($K=7$).
   - Giúp không gian quyết định ($w_1, \dots, w_7$) gọn gàng, tăng tốc độ hội tụ cho **NSGA-III**, và đặc biệt phù hợp để xây dựng không gian đặc trưng hành vi (Behavior Descriptors) trong **MAP-Elites** & **CVT-MAP-Elites**.
2. **Công thức toán học gom nhóm (Equal-Weighted Sector Return)**:
   $$R_{k, t} = \frac{1}{|S_k|} \sum_{i \in S_k} R_{i, t}$$
   *Trong đó: $S_k$ là tập hợp các mã thuộc ngành $k$, $R_{i, t}$ là tỷ suất sinh lợi của mã $i$ tại ngày $t$.*
3. **Thu thập dữ liệu 5 năm (~1.290 phiên giao dịch)**:
   - Xuất file cốt lõi vào thư mục `data/`: `data/VN30_Sectors_Returns_5Years.csv` và `data/VN30_Sector_Mapping.csv`.

In [ ]:
import os
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import yfinance as yf

# 1. Danh sách 30 mã cổ phiếu thành phần VN30
vn30_tickers = [
    "ACB.VN", "BCM.VN", "BID.VN", "BVH.VN", "CTG.VN", "FPT.VN", "GAS.VN", "GVR.VN", 
    "HDB.VN", "HPG.VN", "MBB.VN", "MSN.VN", "MWG.VN", "PLX.VN", "POW.VN", "SAB.VN", 
    "SHB.VN", "SSB.VN", "SSI.VN", "STB.VN", "TCB.VN", "TPB.VN", "VCB.VN", "VHM.VN", 
    "VIB.VN", "VIC.VN", "VJC.VN", "VNM.VN", "VPB.VN", "VRE.VN"
]

# 2. Mốc thời gian 5 năm gần nhất
end_date = datetime.now().strftime('%Y-%m-%d')
start_date = (datetime.now() - timedelta(days=5*365)).strftime('%Y-%m-%d')

print(f"Đang tải dữ liệu 30 mã VN30 từ {start_date} đến {end_date} (5 NĂM)...")
df_prices_raw = yf.download(vn30_tickers, start=start_date, end=end_date, threads=False)['Close']

# Loại bỏ đuôi '.VN' ở tên cột
df_prices_raw.columns = [c.replace('.VN', '') for c in df_prices_raw.columns]
df_prices = df_prices_raw.dropna()

# Tính Daily Returns của 30 mã
returns_30 = df_prices.pct_change().dropna()

print(f"-> Tải thành công {len(returns_30)} ngày giao dịch của {returns_30.shape[1]} mã cổ phiếu.")

## BƯỚC 1: Phân loại 30 mã vào 7 Nhóm Ngành Chuẩn

Dựa trên chuẩn phân ngành ICB và cấu trúc thị trường Việt Nam:
- **NganHang** (13 mã): `ACB, BID, CTG, HDB, MBB, SHB, SSB, STB, TCB, TPB, VCB, VIB, VPB`
- **BatDongSan** (4 mã): `BCM, VHM, VIC, VRE`
- **TieuDung_BanLe** (4 mã): `MWG, MSN, SAB, VNM`
- **NangLuong_TienIch** (3 mã): `GAS, PLX, POW`
- **Thep_VatLieu** (2 mã): `HPG, GVR`
- **CongNghe** (1 mã): `FPT`
- **TaiChinh_DichVu** (3 mã): `SSI, BVH, VJC`

In [ ]:
# Từ điển ánh xạ nhóm ngành
sector_dict = {
    'NganHang': ['ACB', 'BID', 'CTG', 'HDB', 'MBB', 'SHB', 'SSB', 'STB', 'TCB', 'TPB', 'VCB', 'VIB', 'VPB'],
    'BatDongSan': ['BCM', 'VHM', 'VIC', 'VRE'],
    'TieuDung_BanLe': ['MWG', 'MSN', 'SAB', 'VNM'],
    'NangLuong_TienIch': ['GAS', 'PLX', 'POW'],
    'Thep_VatLieu': ['HPG', 'GVR'],
    'CongNghe': ['FPT'],
    'TaiChinh_DichVu': ['SSI', 'BVH', 'VJC']
}

# 1. Áp dụng công thức Gom nhóm (Equal-Weighted Sector Returns)
sector_returns = pd.DataFrame(index=returns_30.index)
for sec_name, t_list in sector_dict.items():
    sector_returns[sec_name] = returns_30[t_list].mean(axis=1)

# 2. Tính Giá chỉ số ngành chuẩn hóa (Base = 100)
sector_prices = (1 + sector_returns).cumprod() * 100

# 3. Tạo bảng ánh xạ Sector Mapping (Metadata)
mapping_rows = []
for sec_name, t_list in sector_dict.items():
    for t in t_list:
        mapping_rows.append({'Ticker': t, 'Sector': sec_name})
sector_mapping = pd.DataFrame(mapping_rows)

print("Hoàn tất tính toán 7 Siêu tài sản ngành!")
display(sector_returns.head())

## BƯỚC 2: Xuất Dataset vào thư mục `data/`

Tất cả các file dataset được gom gọn trong thư mục `data/` để dễ quản lý:

In [ ]:
# Tạo thư mục data/ nếu chưa có
os.makedirs('data', exist_ok=True)

# Lưu các dataset rút gọn theo Cách 1 vào data/
sector_returns.to_csv('data/VN30_Sectors_Returns_5Years.csv')
sector_prices.to_csv('data/VN30_Sectors_Prices_5Years.csv')
sector_mapping.to_csv('data/VN30_Sector_Mapping.csv', index=False)

# Lưu backup dataset 30 mã gốc 5 năm vào data/ (để đối chiếu nếu cần)
returns_30.to_csv('data/VN30_Returns_5Years.csv')
df_prices.to_csv('data/VN30_Prices_5Years.csv')

print("ĐÃ XUẤT CÁC DATASET CỐT LÕI VÀO THƯ MỤC 'data/' THÀNH CÔNG:")
print(f"1. data/VN30_Sectors_Returns_5Years.csv -> {sector_returns.shape} (Dùng chính cho thuật toán PO)")
print(f"2. data/VN30_Sectors_Prices_5Years.csv  -> {sector_prices.shape}  (Giá chuẩn hóa index 100)")
print(f"3. data/VN30_Sector_Mapping.csv         -> {len(sector_mapping)} mã phân ngành")
print(f"4. data/VN30_Returns_5Years.csv         -> {returns_30.shape} (Dữ liệu 30 mã đối chứng)")
print(f"5. data/VN30_Prices_5Years.csv          -> {df_prices.shape}  (Giá 30 mã đối chứng)")

## BƯỚC 3: Thống kê Thuộc tính Tài chính của 7 Nhóm Ngành (5 Năm)

- Lợi nhuận kỳ vọng năm (Annualized Return: $\mu_k = \bar{R}_k \times 252$)
- Rủi ro năm (Annualized Volatility: $\sigma_k = \text{std}(R_k) \times \sqrt{252}$)
- Tỷ số Sharpe ($S_k = \mu_k / \sigma_k$)
- Ma trận tương quan (Correlation Matrix)

In [ ]:
ann_return = sector_returns.mean() * 252
ann_volatility = sector_returns.std() * np.sqrt(252)
sharpe = ann_return / ann_volatility

stats_df = pd.DataFrame({
    'Nhóm ngành': sector_returns.columns,
    'Số mã trong nhóm': [len(sector_dict[s]) for s in sector_returns.columns],
    'Lợi nhuận năm (%)': (ann_return * 100).round(2).values,
    'Rủi ro biến động (%)': (ann_volatility * 100).round(2).values,
    'Sharpe Ratio': sharpe.round(3).values
}).set_index('Nhóm ngành')

print("BẢNG THỐNG KÊ 7 NHÓM NGÀNH VN30 (5 NĂM):")
display(stats_df.sort_values(by='Sharpe Ratio', ascending=False))

print("\nMA TRẬN HỆ SỐ TƯƠNG QUAN (CORRELATION MATRIX):")
display(sector_returns.corr().round(3))